In [1]:
! pip install --upgrade --q pinecone-client pinecone-text pinecone-notebook

ERROR: Ignored the following versions that require a different python version: 0.3.2 Requires-Python >=3.8,<3.11; 0.3.3 Requires-Python >=3.8,<3.11; 0.3.4 Requires-Python >=3.8,<3.11; 0.4.0 Requires-Python >=3.8,<3.11; 0.4.1 Requires-Python >=3.8,<3.11; 2.2.5 Requires-Python >=3.9,<3.13; 2.2.5rc2 Requires-Python >=3.9,<3.13; 2.3.0.dev1 Requires-Python >=3.9,<3.13; 3.0.0 Requires-Python >=3.8,<3.13; 3.0.0.dev1 Requires-Python >=3.9,<3.13; 3.0.0.dev10 Requires-Python >=3.8,<3.13; 3.0.0.dev2 Requires-Python >=3.9,<3.13; 3.0.0.dev3 Requires-Python >=3.9,<3.13; 3.0.0.dev4 Requires-Python >=3.9,<3.13; 3.0.0.dev5 Requires-Python >=3.9,<3.13; 3.0.0.dev6 Requires-Python >=3.9,<3.13; 3.0.0.dev7 Requires-Python >=3.8,<3.13; 3.0.0.dev8 Requires-Python >=3.8,<3.13; 3.0.0.dev9 Requires-Python >=3.8,<3.13; 3.0.0rc3 Requires-Python >=3.8,<3.13; 3.0.1 Requires-Python >=3.8,<3.13; 3.0.2 Requires-Python >=3.8,<3.13; 3.0.3 Requires-Python >=3.8,<3.13; 3.1.0.dev1 Requires-Python >=3.8,<3.13
ERROR: Could no

In [13]:
import os 
from dotenv import load_dotenv
load_dotenv()



api_key =  os.getenv("PINECONE_API_KEY")

PineHybridSearchRetriever is class used to create retriever both semantic search and syntatic search


In [14]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

* create index in pinecone

In [15]:
import os
from pinecone import Pinecone, ServerlessSpec

# initialize the pinecone client
index_name = "rag-hybrid-search-langchain"
pc = Pinecone(api_key)

#create the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name = index_name,
        dimension=384, #dimension of dense vector. 384 is used because in hugging face embedding by default it is 384
        metric='dotproduct', ## sparse values dupported only for dotproduct
        spec = ServerlessSpec(cloud='aws', region="us-east-1")
    )

In [16]:
index=pc.Index(index_name)
index

In [18]:
# vector embedding and sparse matrix
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings

os.environ['HF_TOKEN'] = os.getenv("HF_TOKEN")

embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 149.67it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [30]:
# embeddings.embed_query("What is hybrid search?")

In [20]:
from pinecone_text.sparse import BM25Encoder


In [21]:
bm25_encoder = BM25Encoder().default() # by default bm_25 uses tfidf technique and is responsible in doing sparse matrix conversion for any queries
bm25_encoder

In [22]:
sentences = [
    "charcol is black in color.",
    "charcol is not white in color.",
    "leaves are green in color.",
    "water is colorless."
]

# now apply tfidf values on these senternces
bm25_encoder.fit(sentences)

# now store the values to a json file
bm25_encoder.dump("bm25_values.json")

# load to your BM25Encoder object
bm25_encoder = BM25Encoder().load("bm25_values.json")

## above whole process in this cell is done for sparse matrix. and sparse matrix created

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 332.72it/s]


In [23]:
# lets create retriever. so that we can get combination of both dense and sparse.
from langchain_community.retrievers import PineconeHybridSearchRetriever
retriever=PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25_encoder,index=index)

In [27]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000001D9EB530710>, index=<pinecone.db_data.index.Index object at 0x000001D9D4C21340>)

In [28]:
# add all the text that all we want. which will inside my index
retriever.add_texts(
    [
    "charcol is black in color.",
    "charcol is not white in color.",
    "leaves are green in color.",
    "water is colorless."
]
)

100%|██████████| 1/1 [00:04<00:00,  4.14s/it]


In [31]:
print(index.describe_index_stats())

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '185',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 01 Mar 2026 18:17:18 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '39',
                                    'x-pinecone-request-latency-ms': '38',
                                    'x-pinecone-response-duration-ms': '41'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'__default__': {'vector_count': 4}},
 'storageFullness': 0.0,
 'total_vector_count': 4,
 'vector_type': 'dense'}


In [35]:
retriever.invoke('what is color of charcol ')

[Document(metadata={'score': 0.641871333}, page_content='charcol is not white in color.'),
 Document(metadata={'score': 0.637722135}, page_content='charcol is black in color.'),
 Document(metadata={'score': 0.268456548}, page_content='leaves are green in color.'),
 Document(metadata={'score': 0.18088007}, page_content='water is colorless.')]

In [24]:
import sys
print(sys.executable)

d:\RAG\rag---hybrid-search-\ragenv312\Scripts\python.exe
